In [0]:
import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

def save_to_csv(df, save_path):
    (df.coalesce(1)
        .write.format('csv')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save csv to ", save_path)

In [0]:
par_month = [
    # '202601'
            #  '202602'
            #  ,'202603'
            '202604',
            '202605',
            '202606'
             ]
exclude_cols = {"latitude", "longitude"}
numeric_types = {"integer"}
for i in par_month:
    volume_path = f"dbfs:/Volumes/int-cu-siampiwat/staging/report/report4/{i}/report4/"
    path_mask = f'dbfs:/Volumes/int-cu-siampiwat/staging/report/report4/mask/{i}/'
    files = dbutils.fs.ls(volume_path)
    report1_files = [file.name for file in files]
    for file in report1_files:
        df = spark.read.csv(volume_path+file, header=True, inferSchema=True)
        
        if file.startswith('report4_bmr'):
            df = df\
                .drop('home_work_unidentified')\
                .withColumnRenamed('other_placetype','home_work_unidentified')

        cols_to_update = [
            field.name for field in df.schema.fields
            if field.dataType.typeName() in numeric_types and field.name not in exclude_cols]

        for col in cols_to_update:
            df = df.withColumn(col, F.when(F.col(col) < 25, 25).otherwise(F.col(col)))

        save_to_csv(df, path_mask+file)